# Demografía electoral de Sonora

Distribución por **sexo y edad** de la lista nominal (`ln`, personas
registradas para votar) y de la participación (`sv`/`lns`), a nivel
**distrito local**, **municipio** y **sección**.

Fuente: conteos censales electorales del INE 2009–2024
(`data/processed/conteos_sonora.csv.gz`, generado por
`00_preparacion_datos`). Cada fila es sección × sexo × edad con la lista
nominal del cuadernillo y las marcas de voto.

> Los conteos solo cubren población **≥ 18 años** (padrón electoral);
> para menores véase el censo INEGI en la libreta `04`.

In [ ]:
import json

import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from vothmo.config import FOLDER_DATA

PROCESSED = FOLDER_DATA / "processed"
CENTRO_SONORA = {"lat": 29.3, "lon": -110.9}
ETIQUETA_SEXO = {0: "Hombres", 1: "Mujeres", 2: "No binario"}
COLOR_SEXO = {"Hombres": "#4C78A8", "Mujeres": "#E45756",
              "No binario": "#72B7B2"}

conteos = pd.read_csv(PROCESSED / "conteos_sonora.csv.gz")
piv = pd.read_csv(PROCESSED / "seccion_distrito_por_anio.csv")

conteos.head()

def geojson_ligero(gdf: gpd.GeoDataFrame) -> dict:
    """GeoJSON con solo geometría e id (los atributos van en los datos)."""
    feats = json.loads(gdf.to_json())["features"]
    for i, f in enumerate(feats):
        f["id"] = i
        f["properties"] = {}
    return {"type": "FeatureCollection", "features": feats}


## 1. Pirámide de la lista nominal de Sonora

Evolución de la estructura por edad y sexo en cada elección
(desliza el año). En 2024 aparece la categoría *No binario*.

In [ ]:
pir = (conteos.groupby(["aelec", "sexo", "edad"], as_index=False)[["ln", "sv"]]
       .sum())
pir["sexo_txt"] = pir["sexo"].map(ETIQUETA_SEXO)
pir["ln_neg"] = pir["ln"] * pir["sexo"].map({0: -1, 1: 1, 2: 0.01})
pir["sv_neg"] = pir["sv"] * pir["sexo"].map({0: -1, 1: 1, 2: 0.01})

fig = px.bar(
    pir, x="ln_neg", y="edad", color="sexo_txt",
    animation_frame="aelec", orientation="h",
    color_discrete_map=COLOR_SEXO,
    range_x=[-pir["ln"].max() * 1.05, pir["ln"].max() * 1.05],
    labels={"ln_neg": "Lista nominal", "edad": "Edad (años)",
            "sexo_txt": "Sexo"},
    title="Pirámide de la lista nominal de Sonora por elección",
)
fig.update_layout(
    barmode="relative", bargap=0.05,
    xaxis=dict(tickformat=",.0f",
               tickvals=[-8000, -4000, 0, 4000, 8000],
               ticktext=["8,000", "4,000", "0", "4,000", "8,000"]),
    height=600,
)
fig.show()

## 2. Por distrito local, municipio o sección

La columna `del` del conteo mapea cada sección a su distrito local (marco
DERFE). Selecciona el nivel y el año en las variables de abajo.

In [ ]:
ANIO = 2024          # año electoral
NIVEL = "del"        # 'del' (distrito local), 'mpionom' (municipio) o 'seccion'

dat = conteos[conteos["aelec"] == ANIO]
unidades = sorted(dat[NIVEL].dropna().unique())
len(unidades), unidades[:10]

In [ ]:
# Pirámides en facetas: una figura por unidad del nivel elegido.
# Para 'seccion' se muestran solo las 12 con mayor lista nominal.
top = (dat.groupby(NIVEL)["ln"].sum()
       .sort_values(ascending=False))
sel = top.index if NIVEL != "seccion" else top.index[:12]

p = (dat[dat[NIVEL].isin(sel)]
     .groupby([NIVEL, "sexo", "edad"], as_index=False)["ln"].sum())
p["sexo_txt"] = p["sexo"].map(ETIQUETA_SEXO)
p["ln_neg"] = p["ln"] * p["sexo"].map({0: -1, 1: 1, 2: 0.01})
p[NIVEL] = p[NIVEL].astype(str)

fig = px.bar(
    p, x="ln_neg", y="edad", color="sexo_txt",
    facet_col=NIVEL, facet_col_wrap=4, orientation="h",
    color_discrete_map=COLOR_SEXO,
    labels={"ln_neg": "Lista nominal", "edad": "Edad", "sexo_txt": "Sexo"},
    title=f"Lista nominal por edad y sexo · {NIVEL} · {ANIO}",
)
fig.update_layout(barmode="relative", bargap=0.05, height=1400)
fig.update_xaxes(tickformat="~s", matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## 3. Participación ciudadana por edad

`pc = sv / lns`: proporción de registros válidos que votaron, por edad y
unidad territorial. Útil para ver en qué edades vota más gente.

In [ ]:
# Participación promedio por edad a nivel estatal, con evolución por año
pc_edad = (conteos[conteos["lns"] > 0]
           .groupby(["aelec", "edad", "sexo"], as_index=False)
           .agg(sv=("sv", "sum"), lns=("lns", "sum")))
pc_edad["pc"] = pc_edad["sv"] / pc_edad["lns"] * 100
pc_edad["sexo_txt"] = pc_edad["sexo"].map(ETIQUETA_SEXO)

fig = px.line(
    pc_edad, x="edad", y="pc", color="sexo_txt",
    animation_frame="aelec",
    color_discrete_map=COLOR_SEXO,
    range_y=[0, 100],
    labels={"pc": "Participación (%)", "edad": "Edad", "sexo_txt": "Sexo"},
    title="Participación ciudadana por edad y sexo en Sonora",
)
fig.update_layout(height=500)
fig.show()

## 4. Distribución espacial

Indicadores por sección sobre la cartografía del año electoral (o el mapa
más cercano): proporción de mujeres en la lista nominal, edad mediana
ponderada y participación. Cambia `ANIO_MAPA` arriba para otros años.

In [ ]:
def mapa_para_anio(anio: int) -> int:
    years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
    return min(years, key=lambda y: abs(y - anio))


dat = conteos[conteos["aelec"] == ANIO]
sec = (dat.groupby(["seccion", "sexo"], as_index=False)
       .agg(ln=("ln", "sum"), lns=("lns", "sum"), sv=("sv", "sum"),
            edad_ln=("edad", "count")))

sec_tot = sec.groupby("seccion", as_index=False)["lns"].sum()
sec_muj = (sec[sec["sexo"] == 1][["seccion", "lns"]]
           .rename(columns={"lns": "lns_muj"}))
edad_med = (dat[dat["lns"] > 0]
            .groupby("seccion")
            .apply(lambda d: (d["edad"] * d["lns"]).sum() / d["lns"].sum(),
                   include_groups=False)
            .rename("edad_med").reset_index())
pc_sec = (dat.groupby("seccion", as_index=False)
          .agg(sv=("sv", "sum"), lns=("lns", "sum")))
pc_sec["pc"] = pc_sec["sv"] / pc_sec["lns"] * 100

g = gpd.read_file(PROCESSED / f"secciones_{mapa_para_anio(ANIO)}.gpkg")
g = (g.merge(sec_tot, on="seccion", how="left")
      .merge(sec_muj, on="seccion", how="left")
      .merge(edad_med, on="seccion", how="left")
      .merge(pc_sec[["seccion", "pc"]], on="seccion", how="left"))
g["pct_muj"] = g["lns_muj"] / g["lns"] * 100
g.head()

In [ ]:
INDICADOR = "pct_muj"   # 'pct_muj' | 'edad_med' | 'pc'
TITULOS = {"pct_muj": "Mujeres en la lista nominal (%)",
           "edad_med": "Edad mediana de la lista nominal",
           "pc": "Participación ciudadana (%)"}

fig = px.choropleth_map(
    g, geojson=geojson_ligero(g), locations=g.index,
    color=INDICADOR, color_continuous_scale="RdBu_r",
    map_style="carto-positron", center=CENTRO_SONORA, zoom=4.6,
    hover_data={"seccion": True, "municipio": True, "distrito_l": True,
                INDICADOR: ":.1f"},
    title=f"{TITULOS[INDICADOR]} por sección · {ANIO}",
)
fig.update_traces(marker_line_width=0.2, marker_line_color="white")
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0), height=650)
fig.show()

In [ ]:
# Los mismos indicadores agregados por distrito local (mapa)
dat = conteos[conteos["aelec"] == ANIO].dropna(subset=["del"]).copy()
dat["del"] = dat["del"].astype(int)

tot = dat.groupby("del", as_index=False).agg(sv=("sv", "sum"),
                                             lns=("lns", "sum"))
tot["pc"] = tot["sv"] / tot["lns"] * 100
muj = (dat[dat["sexo"] == 1].groupby("del")["lns"].sum()
       .rename("lns_muj").reset_index())
tot = tot.merge(muj, on="del")
tot["pct_muj"] = tot["lns_muj"] / tot["lns"] * 100

gd = gpd.read_file(PROCESSED / f"distritos_simp_{mapa_para_anio(ANIO)}.gpkg")
gd = gd.merge(tot, left_on="distrito_l", right_on="del", how="left")
gd = gd.reset_index(drop=True)

fig = px.choropleth_map(
    gd, geojson=geojson_ligero(gd), locations=gd.index,
    color="pc", color_continuous_scale="Viridis",
    map_style="carto-positron", center=CENTRO_SONORA, zoom=4.6,
    hover_data={"distrito_l": True, "pc": ":.1f", "pct_muj": ":.1f",
                "lns": True},
    title=f"Participación ciudadana por distrito local · {ANIO}",
)
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0), height=650)
fig.show()

## Notas

- `SEXO`: 0 hombres, 1 mujeres, 2 no binario (únicamente en 2024).
- `lns = ln - ns`: registros válidos (lista nominal sin registros no
  especificados); `pc = sv/lns` es la participación usada aquí.
- La agregación por distrito usa `del` del propio conteo (marco DERFE
  armo­nizado); para 2024 los valores faltantes ya fueron imputados en
  `00_preparacion_datos` con la cartografía del mismo año.
- Las edades de los conteos son años cumplidos el día de la elección.